# Libs

In [ ]:
import pandas as pd
import numpy as np
import os

import matplotlib.pyplot as plt
import plotly.graph_objects as go
from plotly.subplots import make_subplots


from src.MSCVAE import MSCVAE
from src.MSCRED import MSCRED
from src.ANN_AE import ANN_AE
from src.LSTM_AE import LSTM_AE
from src.CNN_AE import CNN_AE
from src.USAD import USAD
from src.OmniAnomaly import OmniAnomaly
from src.TranAD import TranAD

DIR_DATA = os.getcwd()+"/data/"

# Load Data

In [ ]:
class_col = "CLASS"

base_name_normal = "normal.csv"
base_name_falha2 = "falha2.csv"
base_name_falha7 = "falha7.csv"
base_name_falha15 = "falha15.csv"

In [ ]:
df_normal = pd.read_csv(DIR_DATA + base_name_normal, sep=";", decimal=".")
df_normal_labels = df_normal[class_col]
df_normal.drop(class_col, axis=1, inplace=True)
df_normal

In [ ]:
df_falha2 = pd.read_csv(DIR_DATA + base_name_falha2, sep=";", decimal=".")
df_falha2_labels = df_falha2[class_col]
df_falha2.drop(class_col, axis=1, inplace=True)
df_falha2

In [ ]:
df_falha7 = pd.read_csv(DIR_DATA + base_name_falha7, sep=";", decimal=".")
df_falha7_labels = df_falha7[class_col]
df_falha7.drop(class_col, axis=1, inplace=True)
df_falha7

In [ ]:
df_falha15 = pd.read_csv(DIR_DATA + base_name_falha15, sep=";", decimal=".")
df_falha15_labels = df_falha15[class_col]
df_falha15.drop(class_col, axis=1, inplace=True)
df_falha15

In [ ]:
df_sistema = pd.read_csv(DIR_DATA + "CSTR_subsistema.csv", sep=";")
df_sistema

# Fit

In [ ]:
mscvae = MSCVAE()

gain = 1
epochs = 20
mscvae.fit([df_normal], gain=gain, epochs=epochs, verbose=True)

# Predict

In [ ]:
predictions_mscvae = mscvae.predict(df_falha15)

In [ ]:
def plot_predict(predictions, threshold):
    # Criar o dataframe extraindo diretamente da sua estrutura de saída
    df = pd.DataFrame({
        "timestamp": predictions['timestamp'],
        "phi": predictions['phi']/threshold
    })

    df["threshold"] = threshold/threshold

    # Configurar o layout
    layout = go.Layout(plot_bgcolor="white", paper_bgcolor='white')
    fig = go.Figure(layout=layout)
    
    # Adicionar a série do Índice (Phi)
    fig.add_trace(go.Scatter(
        x=df["timestamp"], 
        y=df["phi"], 
        mode="lines", 
        name="Índice (Phi)", 
        fill="tozeroy", 
        line_color="#0F293A"
    ))
    
    # Adicionar a linha do Limiar
    fig.add_trace(go.Scatter(
        x=df["timestamp"], 
        y=df["threshold"], 
        mode="lines", 
        name="Limiar", 
        line_color="#FB8102"
    ))
    
    # Ajustar eixos
    fig.update_xaxes(showgrid=False)
    fig.update_yaxes(showgrid=True, gridwidth=0.5, gridcolor='#CECFD1')
    
    # Ajustar layout final
    fig.update_layout(
        hovermode='x unified', 
        legend=dict(orientation="h"),
        yaxis_range=[0, 4 * threshold/threshold] # Mantém o zoom vertical focado no limite inferior
    )

    return fig

fig = plot_predict(predictions_mscvae, threshold=mscvae.threshold)
fig.show()

# Contribution

In [ ]:
contribution, df_reconstruction = mscvae.contribution(df_falha15, df_sistema)
pd.DataFrame().from_dict(contribution).head(10)